In [12]:
# Example code using PyTorch and torchvision
import torch
from torchvision import models, transforms
from PIL import Image
import os
import numpy as np
import cv2

# Define the denormalize function
def denormalize(
    x,
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
):
    """
    Denormalizes a tensor image.

    :param x: The normalized image tensor.
    :param mean: The mean used for normalization.
    :param std: The standard deviation used for normalization.
    :return: The denormalized image tensor.
    """
    # Ensure the tensor is on the CPU before denormalization if it's on CUDA
    x = x.cpu()
    for t, m, s in zip(x, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(x, 0, 1) # Clamp values to be within [0, 1]

# Load pre-trained ResNet model
resnet_model = models.resnet18(pretrained=False)

# Get the number of features in the last layer
num_ftrs = resnet_model.fc.in_features
# Replace the final fully connected layer to match the number of classes in your checkpoint
# Based on the error message, your checkpoint was trained for 2 classes
resnet_model.fc = torch.nn.Linear(num_ftrs, 2)

# Load the checkpoint
checkpoint = torch.load('/content/drive/MyDrive/ResNetModel/outputs/model.pth', map_location=torch.device('cpu'), weights_only=False)

# Extract the model's state dictionary from the checkpoint
# The error message indicates that the model's state_dict is stored under the key 'model_state_dict'
model_state_dict = checkpoint['model_state_dict']

# Load the model's state dictionary into the model
resnet_model.load_state_dict(model_state_dict)

resnet_model.eval()

# Preprocess input image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def save_test_results(
    tensor,
    output_class,
    output_confidence,
    counter,
    test_result_save_dir
):
    """
    This function will save a few test images along with the
    ground truth label and predicted label annotated on the image.

    :param tensor: The image tensor.
    :param target: The ground truth class number.
    :param output_class: The predicted class number.
    :param counter: The test image number.
    """
    # Denormalize the tensor before converting to numpy and saving
    image = denormalize(tensor).cpu()
    # Add a batch dimension if it's missing (e.g., if the input tensor was squeezed)
    if image.ndim == 3:
        image = image.unsqueeze(0)
    image = image.squeeze(0).permute((1, 2, 0)).numpy()
    image = np.ascontiguousarray(image, dtype=np.float32)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    color = (0, 0, 255)
    cv2.putText(
        image, f"Pred: {CLASS_NAMES[int(output_class)]}",
        (5, 25), cv2.FONT_HERSHEY_SIMPLEX,
        0.6, color, 2, cv2.LINE_AA
    )
    cv2.imwrite(
        os.path.join(test_result_save_dir, 'test_image_'+str(counter).zfill(5)+'.png'),
        image*255.
    )

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['OccludedEye', 'OpenEye']
test_result_save_dir = '/content/drive/MyDrive/ResNetModel/outputs/prediction_results2'
numFrames = 24998
List = np.zeros((2, numFrames)) # Changed to 2 rows to store both class and confidence
for i in range(0, numFrames):
  image_path = os.path.join('/content/drive/MyDrive/ResNetModel/input/Frames/Frame000'+str(i).zfill(5)+'.png')
  image = Image.open(image_path)
  # Convert the image to RGB format before applying transformations
  image = image.convert('RGB')
  input_tensor = transform(image)
  input_batch = input_tensor.unsqueeze(0)

  # Make prediction
  with torch.no_grad():
      output = resnet_model(input_batch)

  # Print raw output before softmax
  print(f"Raw output before softmax: {output}")

  # Get predicted class and confidence
  class_confidence = torch.nn.functional.softmax(output, dim=1) # Apply softmax along dimension 1
  _, predicted_class = torch.max(class_confidence, 1) # Get the predicted class index from softmax output
  predicted_confidence_score = class_confidence[0, predicted_class.item()].item() # Get the confidence score for the predicted class


  print(f"Predicted class index: {CLASS_NAMES[predicted_class.item()]} Class Confidence: {predicted_confidence_score} Frame {i+1}/{numFrames}")
  List[0][i] = predicted_class.item()
  List[1][i] = predicted_confidence_score # Store confidence in the second row
  save_test_results(
                  input_tensor,
                  predicted_class.item(),
                  predicted_confidence_score,
                  i,
                  test_result_save_dir
              )
print(List)
# Specify the filename and delimiter for the CSV file
filename = "/content/drive/MyDrive/ResNetModel/outputs/prediction_results2/predictions.csv"
delimiter = ","

# Save the array to the CSV file
np.savetxt(filename, List.T, delimiter=delimiter) # Transpose the list before saving to have columns as class and confidence

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Streaming output truncated to the last 5000 lines.
Raw output before softmax: tensor([[-2.7687,  3.0187]])
Predicted class index: OpenEye Class Confidence: 0.9969431757926941 Frame 22500/24998
Raw output before softmax: tensor([[-3.3302,  3.7057]])
Predicted class index: OpenEye Class Confidence: 0.9991210103034973 Frame 22501/24998
Raw output before softmax: tensor([[-2.7357,  3.5177]])
Predicted class index: OpenEye Class Confidence: 0.9980796575546265 Frame 22502/24998
Raw output before softmax: tensor([[-3.1745,  4.1865]])
Predicted class index: OpenEye Class Confidence: 0.9993647933006287 Frame 22503/24998
Raw output before softmax: tensor([[-2.7700,  3.9769]])
Predicted class index: OpenEye Class Confidence: 0.9988269209861755 Frame 22504/24998
Raw output before softmax: tensor([[-2.9596,  3.7993]])
Predicted class index: OpenEye Class Confidence: 0.9988407492637634 Frame 22505/24998
Raw output before softmax: tensor([[-2.5750,  3.6879]])
Predicted class index: OpenEye Class Conf